In [0]:
df = (spark.table("samples.wanderbricks.payments")
    .filter("payment_method = 'apple_pay'")
    .groupBy("status")
    .count()
)

In [0]:
# Display the physical plan of the DataFrame computation in a simple format
df.explain(mode='simple')

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   HashAggregate(keys=[status#26], functions=[finalmerge_count(merge count#38L) AS count(1)#36L])
   +- Exchange hashpartitioning(status#26, 200), ENSURE_REQUIREMENTS, [plan_id=49]
      +- HashAggregate(keys=[status#26], functions=[partial_count(1) AS count#38L])
         +- Project [status#26]
            +- Filter (isnotnull(payment_method#25) AND (payment_method#25 = apple_pay))
               +- FileScan parquet samples.wanderbricks.payments[payment_method#25,status#26] Batched: true, DataFilters: [isnotnull(payment_method#25), (payment_method#25 = apple_pay)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-west-2-uc-metastore-bucket/metastore/8..., PartitionFilters: [], PushedFilters: [IsNotNull(payment_method), EqualTo(payment_method,apple_pay)], ReadSchema: struct<payment_method:string,status:string>


== Optimizer Statistics (table names per statistics state) ==

In [0]:
# Display the extended physical plan of the DataFrame computation.
# Note: At this stage, AdaptiveSparkPlan will show isFinalPlan=false since execution hasn't been triggered yet.
df.explain(mode='extended')

== Parsed Logical Plan ==
'Aggregate ['status], ['status, 'count(1) AS count#29]
+- Filter (payment_method#25 = apple_pay)
   +- SubqueryAlias samples.wanderbricks.payments
      +- Relation samples.wanderbricks.payments[payment_id#22L,booking_id#23L,amount#24,payment_method#25,status#26,payment_date#27] parquet

== Analyzed Logical Plan ==
status: string, count: bigint
Aggregate [status#26], [status#26, count(1) AS count#29L]
+- Filter (payment_method#25 = apple_pay)
   +- SubqueryAlias samples.wanderbricks.payments
      +- Relation samples.wanderbricks.payments[payment_id#22L,booking_id#23L,amount#24,payment_method#25,status#26,payment_date#27] parquet

== Optimized Logical Plan ==
Aggregate [status#26], [status#26, count(1) AS count#29L]
+- Project [status#26]
   +- Filter (isnotnull(payment_method#25) AND (payment_method#25 = apple_pay))
      +- Relation samples.wanderbricks.payments[payment_id#22L,booking_id#23L,amount#24,payment_method#25,status#26,payment_date#27] parquet

== 

In [0]:
# AQE must finalize the plan before codegen is visible.
# Trigger execution with .collect() first, then inspect codegen.
df.collect()
df.explain(mode="codegen")

Found 2 WholeStageCodegen subtrees.
== Subtree 1 / 2 (maxMethodCodeSize:618; maxConstantPoolSize:323(0.49% used); numInnerClasses:2) ==
*(1) HashAggregate(keys=[status#26], functions=[partial_count(1) AS count#38L], output=[status#26, count#38L])
+- *(1) Project [status#26]
   +- *(1) Filter (isnotnull(payment_method#25) AND (payment_method#25 = apple_pay))
      +- *(1) ColumnarToRow
         +- FileScan parquet samples.wanderbricks.payments[payment_method#25,status#26] Batched: true, DataFilters: [isnotnull(payment_method#25), (payment_method#25 = apple_pay)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-west-2-uc-metastore-bucket/metastore/8..., PartitionFilters: [], PushedFilters: [IsNotNull(payment_method), EqualTo(payment_method,apple_pay)], ReadSchema: struct<payment_method:string,status:string>

Generated code:
/* 001 */ public Object generate(Object[] references) {
/* 002 */   return new GeneratedIteratorForCodegenStage1(references);
/*

In [0]:
df.show()

+---------+-----+
|   status|count|
+---------+-----+
| refunded|  768|
|completed| 9057|
|   failed|  104|
+---------+-----+

